## ROC/EF Validation — Ligand Preparation

This notebook converts actives and decoys from SMILES/CSV format to 2D SDF,
ready for 3D conformer generation.

**Workflow:**
1. Run this notebook to convert actives/decoys SMILES to 2D SDF format.
2. Run `ligand_preparation_pipeline.ipynb` to generate 3D conformers (select RDKit protonation).
3. Use the resulting 3D SDF files as input for ROC/EF docking validation.

**Input files:**
- BRD4: top 20 actives + 699 property-matched decoys
- BRD2: top 20 actives + 449 property-matched decoys

In [ ]:
!pip install rdkit pandas
import io
import pandas as pd
from rdkit import Chem
from rdkit.Chem import SDWriter
from google.colab import files

# Upload CSV or SMI files (multiple files supported)
print("Please upload your files (.csv or .smi, multiple files supported):")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nProcessing: {filename}")

    try:
        # Auto-detect file format
        if filename.lower().endswith('.smi'):
            # .smi file: whitespace-separated, no header
            df = pd.read_csv(io.BytesIO(uploaded[filename]), sep=r'\s+', header=None)
            df.rename(columns={0: 'SMILES', 1: 'ID'}, inplace=True)
        else:
            # .csv file: comma-separated, with header
            df = pd.read_csv(io.BytesIO(uploaded[filename]))

        print(f"[OK] Loaded {len(df)} compounds.")

        # Generate output filename
        sdf_filename = filename.rsplit('.', 1)[0] + '.sdf'
        writer = SDWriter(sdf_filename)
        n_written, n_failed = 0, 0

        for _, row in df.iterrows():
            # Handle various SMILES column name cases
            smiles_col = 'SMILES' if 'SMILES' in df.columns else 'smiles'
            if smiles_col not in df.columns:
                smiles_str = str(row.iloc[0])
            else:
                smiles_str = str(row[smiles_col])

            mol = Chem.MolFromSmiles(smiles_str)
            if mol is None:
                n_failed += 1
                continue

            # Extract molecule name from available columns
            if 'ID' in df.columns:
                mol_name = str(row['ID'])
            elif 'zincid' in df.columns:
                mol_name = str(row['zincid'])
            elif 'Name' in df.columns:
                mol_name = str(row['Name'])
            else:
                mol_name = f"mol_{n_written+1:04d}"

            mol.SetProp('_Name', mol_name)
            writer.write(mol)
            n_written += 1

        writer.close()
        print(f"[OK] {sdf_filename} converted. Success: {n_written}, Failed: {n_failed}")
        files.download(sdf_filename)

    except Exception as e:
        print(f"[ERROR] Failed to process {filename}: {e}")

print("\nAll files processed.")
